In [2]:
import json
import uproot
from pathlib import Path
from typing import Dict, Set, Tuple

## Determine Data Luminosity

In [3]:
# --- Configuration & Constants ---
DATA_BASE_PATH = Path("/home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root")
METADATA_FILE = Path("data_metadata.json")

LUMINOSITY_DICT = {
    "A": 0.5608, "B": 2.0034, "C": 2.9165, "D": 4.7087,
    "E": 1.5031, "F": 3.4567, "G": 3.8969, "I": 5.8896,
    "K": 2.2366, "L": 6.2230
}

PERIODS = list(LUMINOSITY_DICT.keys())

# Using a set literal is slightly faster and cleaner than set([...])
GRL_SET = {
    297730, 298595, 298609, 298633, 298687, 298690, 298771, 298773, 298862, 
    298967, 299055, 299144, 299147, 299184, 299243, 299584, 300279, 300345, 
    300415, 300418, 300487, 300540, 300571, 300600, 300655, 300687, 300784, 
    300800, 300863, 300908, 301912, 301915, 301918, 301932, 301973, 302053, 
    302137, 302265, 302269, 302300, 302347, 302380, 302391, 302393, 302737, 
    302831, 302872, 302919, 302925, 302956, 303007, 303079, 303201, 303208, 
    303264, 303266, 303291, 303304, 303338, 303421, 303499, 303560, 303638, 
    303832, 303846, 303892, 303943, 304006, 304008, 304128, 304178, 304198, 
    304211, 304243, 304308, 304337, 304409, 304431, 304494, 305380, 305543, 
    305571, 305618, 305671, 305674, 305723, 305727, 305735, 305777, 305811, 
    305920, 306269, 306278, 306310, 306384, 306419, 306442, 306448, 306451, 
    307126, 307195, 307259, 307306, 307354, 307358, 307394, 307454, 307514, 
    307539, 307569, 307601, 307619, 307656, 307710, 307716, 307732, 307861, 
    307935, 308047, 308084, 309375, 309390, 309440, 309516, 309640, 309674, 
    309759, 310015, 310247, 310249, 310341, 310370, 310405, 310468, 310473, 
    310634, 310691, 310738, 310809, 310863, 310872, 310969, 311071, 311170, 
    311244, 311287, 311321, 311365, 311402, 311473, 311481
}

# --- Core Functions ---

def get_processed_runs(base_path: Path) -> Dict[str, Set[int]]:
    """Scans the directory structure to find which runs have been processed per period."""
    processed = {}
    if not base_path.exists():
        print(f"Warning: Base path {base_path} does not exist.")
        return processed

    for period_dir in base_path.iterdir():
        if period_dir.is_dir():
            period = period_dir.name
            processed[period] = {
                int(run_dir.name.replace("run", ""))
                for run_dir in period_dir.iterdir() if run_dir.is_dir()
            }
    return processed

def load_metadata(json_path: Path, periods: list) -> Tuple[Dict[str, Set[int]], Dict[int, int]]:
    """
    Parses metadata JSON in a single pass.
    Returns:
        - A dictionary mapping periods to sets of run numbers.
        - A dictionary mapping run numbers to expected event counts.
    """
    with open(json_path, "r") as f:
        metadata = json.load(f)

    runs_by_period = {p: set() for p in periods}
    events_by_run = {}

    for item in metadata:
        run_num = int(item["run_number"])
        events_by_run[run_num] = item["events"]
        
        for p in periods:
            if item["period"].startswith(p):
                runs_by_period[p].add(run_num)

    return runs_by_period, events_by_run

def count_actual_events(run_path: Path) -> int:
    """Safely opens ROOT files in a run directory and sums the entries."""
    num_events = 0
    for file_path in run_path.iterdir():
        if file_path.is_file(): # Can add `and file_path.suffix == '.root'` if needed
            try:
                # Combining path and tree name in uproot.open is often faster
                num_events += uproot.open(f"{file_path}:CollectionTree").num_entries
            except Exception as e:
                # Catching a bare Exception is okay here, but we log the specific error
                print(f"Error reading {file_path.name}: {e}")
    return num_events

def analyze_period(period: str, processed_runs: Set[int], runs_in_period: Set[int], events_by_run: Dict[int, int]):
    """Calculates actual vs expected events and effective luminosity for a specific period."""
    if period not in LUMINOSITY_DICT:
        print(f"Period {period} not found in Luminosity Dictionary.")
        return

    print(f"\n--- Analyzing Period {period} ---")
    print(f"Total possible luminosity: {LUMINOSITY_DICT[period]}")
    
    available_grl = runs_in_period.intersection(GRL_SET)
    print(f"Runs in period {period} that are in the GRL: {len(available_grl)}")

    runs_processed = processed_runs.intersection(available_grl)
    print(f"Runs in period {period} (GRL + Processed): {len(runs_processed)}")

    print("\nAll Available GRL Runs:")
    print(sorted(available_grl))
    print("="*50)

    total_actual_events = 0
    period_path = DATA_BASE_PATH / period

    # Calculate Actual Events
    for run in sorted(runs_processed):
        run_path = period_path / f"run{run}"
        actual_events = count_actual_events(run_path)
        print(f"{run} \t Actual Events: {actual_events}")
        total_actual_events += actual_events

    print(f"Total actual events for period {period}: {total_actual_events}")

    # Calculate Expected Events
    total_expected_events = 0
    for run in runs_processed:
        expected_events = events_by_run.get(run, 0)
        total_expected_events += expected_events
        print(f"{run} \t Expected Events: {expected_events}")

    print(f"Total expected events for period {period}: {total_expected_events}")
    print("="*50)

    # Calculate Final Luminosity
    if total_expected_events > 0:
        eff_lumi = (total_actual_events / total_expected_events) * LUMINOSITY_DICT[period]
        print(f"Effective Luminosity processed: {eff_lumi:.4f}")
        return eff_lumi
    else:
        print("Effective Luminosity processed: 0.0000 (No expected events found)")
        return 0

In [4]:
processed_runs_dict = get_processed_runs(DATA_BASE_PATH)
runs_by_period, expected_events_by_run = load_metadata(METADATA_FILE, PERIODS)

In [5]:
total_lumi = 0
for p in PERIODS:
    lumi = analyze_period(
        period=p, 
        processed_runs=processed_runs_dict.get(p, set()), 
        runs_in_period=runs_by_period.get(p, set()), 
        events_by_run=expected_events_by_run
    )
    total_lumi = total_lumi + lumi


--- Analyzing Period A ---
Total possible luminosity: 0.5608
Runs in period A that are in the GRL: 17
Runs in period A (GRL + Processed): 17

All Available GRL Runs:
[297730, 298595, 298609, 298633, 298687, 298690, 298771, 298773, 298862, 298967, 299055, 299144, 299147, 299184, 299243, 299584, 300279]
297730 	 Actual Events: 14666349
298595 	 Actual Events: 3133859
298609 	 Actual Events: 6288037
298633 	 Actual Events: 13229931
298687 	 Actual Events: 17971981
298690 	 Actual Events: 1258445
298771 	 Actual Events: 7061356
298773 	 Actual Events: 6837614
298862 	 Actual Events: 20220154
298967 	 Actual Events: 27997456
Error reading root_299055_26.root: not found: 'CollectionTree' (with any cycle number)

    Available keys: (none!)

in file /home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root/A/run299055/root_299055_26.root
Error reading root_299055_54.root: not found: 'CollectionTree' (with any cycle number)

    Available keys: (none!)

in file /home/aegis/ether/Research_H

In [6]:
print("Total Luminosity being processed:", total_lumi)

Total Luminosity being processed: 25.39457671040864


In [16]:
BASE_PATH = "/home/aegis/ether/Research_HEP/Dataset_ver3/MC/reduce_root"

mc_processes = "Diboson", "Multijet", "Single_top", "ttbar", "Wjets", "Zjets"

with open("mc_metadata.json", "r") as f:
    METADATA = json.load(f)


In [23]:
# Process Diboson files
diboson_path = os.path.join(BASE_PATH, "Diboson")

for sub_process in os.listdir(diboson_path):
    sub_process_path = os.path.join(diboson_path, sub_process)
    print(f'Processing {sub_process.replace("mc20_13TeV_MC_", "")} files...')
    cross_section = METADATA["Diboson"][sub_process.replace("mc20_13TeV_MC_", "")]["xsec_pb"]
    print(f"{sub_process} \t Cross-section: {cross_section} pb")
    num_events = 0
    for file in os.listdir(sub_process_path):
        file_path = os.path.join(sub_process_path, file)
        num_events = num_events + uproot.open(file_path)["CollectionTree"].num_entries

    print(f"{sub_process} \t Number of Events: {num_events}")


Processing Sh_2211_WlvZqq files...
mc20_13TeV_MC_Sh_2211_WlvZqq 	 Cross-section: 8.582 pb
mc20_13TeV_MC_Sh_2211_WlvZqq 	 Number of Events: 420000
Processing Sh_2211_WlvWqq files...
mc20_13TeV_MC_Sh_2211_WlvWqq 	 Cross-section: 109.28 pb
mc20_13TeV_MC_Sh_2211_WlvWqq 	 Number of Events: 2251000
Processing Sh_2211_WqqZvv files...
mc20_13TeV_MC_Sh_2211_WqqZvv 	 Cross-section: 6.543 pb
mc20_13TeV_MC_Sh_2211_WqqZvv 	 Number of Events: 409000
Processing Sh_2211_ZqqZvv files...
mc20_13TeV_MC_Sh_2211_ZqqZvv 	 Cross-section: 8.357 pb
mc20_13TeV_MC_Sh_2211_ZqqZvv 	 Number of Events: 300000
